In [1]:
print("hello world")

hello world


In [3]:
!ls /home/aria/Downloads | grep final

finalAchareh.csv


In [6]:
path = "/home/aria/Downloads/finalAchareh.csv"



In [10]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

In [8]:
df = pd.read_csv(path)
df.shape

(400, 17)

In [11]:
data = df[df["Base Category"] == "نظافت و پذیرایی"].copy()

date_cols = [
    "Customer Joined on",
    "Expert Joined on",
    "Order Created at",
    "Order Start Time",
    "Order Finish Time"
]

for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors="coerce")

data = data.sort_values("Order Created at").reset_index(drop=True)

data["Year"] = data["Order Created at"].dt.year
data["Month"] = data["Order Created at"].dt.month
data["Day"] = data["Order Created at"].dt.day
data["Weekday"] = data["Order Created at"].dt.weekday
data["Hour"] = data["Order Created at"].dt.hour

data["Duration Hours"] = (
    data["Order Finish Time"] - data["Order Start Time"]
).dt.total_seconds() / 3600

data["Start Delay Hours"] = (
    data["Order Start Time"] - data["Order Created at"]
).dt.total_seconds() / 3600

In [12]:
target = "Price (Tomans)"

features = [
    "Subcategory",
    "Order Status",
    "Order City",
    "Order Region",
    "Order's Expert Gender Options ",
    "Device",
    "Discount (Tomans)",
    "Year",
    "Month",
    "Day",
    "Weekday",
    "Hour",
    "Duration Hours",
    "Start Delay Hours"
]

X = data[features]
y = data[target]

X_train = X.iloc[:-100]
X_test = X.iloc[-100:]

y_train = y.iloc[:-100]
y_test = y.iloc[-100:]

print(X_train.shape, X_test.shape)

(162, 14) (100, 14)


In [13]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

def make_preprocessor(scale=False):
    numeric_steps = [("fill", SimpleImputer(strategy="median"))]

    if scale:
        numeric_steps.append(("scale", StandardScaler()))

    categorical_steps = [
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ]

    if scale:
        categorical_steps.append(("scale", StandardScaler()))

    return ColumnTransformer([
        ("cat", Pipeline(categorical_steps), categorical_cols),
        ("num", Pipeline(numeric_steps), numeric_cols)
    ], verbose_feature_names_out=False)

In [14]:
linear_model = Pipeline([
    ("prepare", make_preprocessor()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

predictions = linear_model.predict(X_test)

r2 = r2_score(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5

print("R2:", r2)
print("RMSE:", rmse)

R2: 0.401151571168901
RMSE: 317013.47862445435


In [15]:
ridge_model = Pipeline([
    ("prepare", make_preprocessor(scale=True)),
    ("model", Ridge(alpha=100))
])

ridge_model.fit(X_train, y_train)

ridge_predictions = ridge_model.predict(X_test)

print("Ridge R2:", r2_score(y_test, ridge_predictions))
print("Ridge RMSE:", mean_squared_error(
    y_test,
    ridge_predictions
) ** 0.5)

Ridge R2: 0.3511957347185505
Ridge RMSE: 329971.2626744976


In [16]:
feature_names = ridge_model["prepare"].get_feature_names_out()
coefficients = ridge_model["model"].coef_

importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Importance": abs(coefficients)
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance.head(3)

,Feature,Coefficient,Importance
12,Duration Hours,159499.281906,159499.281906
2,Order City,-40427.839134,40427.839134
3,Order Region,-30966.694587,30966.694587
